In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC


In [ ]:
DATA_DIR = Path("data")
DATA_FILE = DATA_DIR / "diabetes_prediction_dataset.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Run this notebook from its folder "
        "or update DATA_DIR to the sibling data folder."
    )
df = pd.read_csv(DATA_FILE)
target = "diabetes"
numeric_features = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
]
categorical_features = ["gender", "smoking_history"]
features = numeric_features + categorical_features
required_columns = features + [target]
if missing := set(required_columns) - set(df.columns):
    raise ValueError(f"Dataset is missing columns: {sorted(missing)}")
if df[target].isna().any() or set(df[target].unique()) != {0, 1}:
    raise ValueError("diabetes must contain both binary labels 0 and 1 without missing values")
X = df[features]
y = df[target].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print(f"Rows: {len(df)}; training rows: {len(X_train)}; test rows: {len(X_test)}")
print("Target counts:", y.value_counts().sort_index().to_dict())
df.head()


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)
# LinearSVC scales much better than kernel SVC on this 100,000-row dataset.
svm_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LinearSVC(C=1.0, class_weight="balanced", random_state=42, max_iter=5000),
        ),
    ]
)
svm_model.fit(X_train, y_train)
print("SVM training complete.")


In [ ]:
y_pred = svm_model.predict(X_test)
y_scores = svm_model.decision_function(X_test)
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_scores):.3f}")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"]))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No diabetes", "Diabetes"],
    cmap="Blues",
)
plt.title("Linear SVM - diabetes classification")
plt.tight_layout()
plt.show()
